# 导言

## what can we learn in this class:
- mechanics:how things work (transformer是什么，模型如何并行运算)
- mindset:如何最大限度发挥硬件的潜能，同时重视可扩展性问题
- intuitions:什么样的数据和模型决策可以得到最高的准确性

# LLM历史


## pre-neural(?-2010)
- shannon:测量英语信息熵的语言模型
- N-gram语言模型
## neural ingredients神经网络的组成要素(2010初期)
- LSTM
- 2003Bengio:第一个神经语言模型
- 2014sequence2sequence(for machine translation)
- 2014adam优化器
- 2014bahdanau attention mechnism
- 2017 transformer
- 2017MoE(Mixture of Expert混合专家)
- 20182019模型并行运算
## early foundation model（2010末期）
- ELMo
- BERT
- google T5（11B）
## scaling model
- 2019:GPT2（1.5B）
- Kaplan2020:scaling law
- 2020:GPT3(175B)
- 2022:PaLM(540B)

## open models
- early attempt
	- eleuther：一个复现并开源GPT3的任务
	- meta
	- hugging face
- credible open-weight models(weights+paper)
	- meta Llama
	- deepseek
	- alibaba qwen
	- moonshot kimi
	- GLM
	- minimax
	- xiaomi MIMO
- open-source models(weights+paper+code+data)
	- Olmo
	- NVDIA Nemotron
	- marin

# 课程大纲

- basics：从0开始训练一个语言模型
	- tokennization：分词
		- token：模型处理的基本单位
		- tokenizer：分词器，在原始输入和向量之间进行转化
		- 经典分词器：BPE（Byte-Pair Encoding）
		- 把原始bytes流构成的长序列缩短为更少的tokens
		- 可以进行自适应计算
	- model architecture
		- transformer
			- 基本结构
			- 激活函数的变化
			- 位置编码、归一化做法的改变
			- 防止数值爆炸的做法的改进
			- 全注意力机制改为多种方法来减少注意力计算量（前沿解法：状态空间模型/线性注意力机制）
			- MoE已称为构建高效transformer的主要范式
			- shape(隐藏层维度，层深，多头注意力数量，专家数量)
	- training
		- 损失函数（从逐个预测token进化到一次预测多个token，有助于提升模型效果）
		- 优化器（adam→adamW，SOAP,==Muon==）
		- 初始化
		- 学习率
		- 批大小
		- 正则化
		- MoE speicific



# 作业说明


- assignment1：
	- 实现BPE,transformer,loss ,adamW,trainingloop 
	- 搞清楚自己的算力（FLOPs）都花在哪里
		- 浮点运算花在哪里
		- 内存消耗在哪里
		- 在10e12token上训练模型的浮点运算次数=
	- 平衡以下几点：
		- 模型有足够的表达能力（捕捉数据之间的复杂依赖关系）
		- 训练稳定性（参数和梯度范数不太大也不太小）
		- 模型效率（训练和推理效率）
- as2：
	- kernel
	- 多GPU并行
	- inference




# 分词器


## 简介
- 文本即unicode字符串。语言模型可以对token序列建立概率分布，通常用索引表示。所以：需要一个双向转换工具，把字符串编码为token，再把token解码为字符串
- compression ratio压缩比：字节数/token数。注意力序列是n平方复杂度。
- 可以通过扩大词表大小来提高压缩比
	- 但是会引发稀疏性问题，因为词表中的每个词都是一个独立元素
## 如何构建分词器
1. 处理unicode字符串 ，NLP领域过去常用两种做法：unicode和utf-8

unicode字符有15万个（东亚汉字文化圈约十万；还有印非美埃楔等历史文字，符号和emoji，同一字符的重复编码，控制符格式组合符等）

In [33]:
# ord函数可以输入长度为1的字符，输出unicode编码
print(ord("a"))
print(ord("🌏"))
print(ord("❤"))

# 为ord函数写一个异常检测
import warnings # warnings用于发出警告，防止程序直接崩溃
def safe_ord(s):
    if not isinstance(s,str): # isistance用于判断s是否为字符串类型
        raise TypeError("safe_ord只接受字符串") # 如果函数输入不是字符串就抛出typeerror
                                                # raise是抛出异常的意思，程序会在此处停止，并显示该错误
    try:return ord(s)   # try表示尝试执行如下代码
    except TypeError:  # except TypeError用于捕捉try的运行中发生的typeerror。
                        # 如果ord因为s不是单个字符而无法执行，就会跳到except，程序不会崩溃。
        warnings.warn(f"{s!r}不是单个字符，将拆分为{len(s)}个字符分别输出编码")
                        # warnings.warn发出警告但不会让程序崩溃
        return [ord(ch) for ch in s]
print(safe_ord("爱"))
print(safe_ord("爱心"))
# print(safe_ord(789))

97
127759
10084
29233
[29233, 24515]


C:\Users\24310\AppData\Local\Temp\ipykernel_3172\3151290821.py:15: UserWarning: '爱心'不是单个字符，将拆分为2个字符分别输出编码
  warnings.warn(f"{s!r}不是单个字符，将拆分为{len(s)}个字符分别输出编码")


In [32]:
# bytes函数可以输入字符串，输出utf-8编码
print(bytes("d",encoding="utf-8"))
print(bytes("你",encoding="utf-8"))
print(bytes("🌏",encoding="utf-8"))
# utf-8编码用多个0到255间的数字表示一个字符，可用较小的词表表示较长的字节序列

b'd'
b'\xe4\xbd\xa0'
b'\xf0\x9f\x8c\x8f'


## BPE方法
- 最初为做数据压缩，早于语言模型出现；最早被引入NLP，进行神经机器翻译；GPT2首次将BPE用于LLM
- 思路：用原始文本训练分词器，构建出为数据量身定制的词表。
    - 特性：任何内容都可被分词，遇到罕见序列会拆分为一个更小的单元。

### 构建一个最小可运行版本：

In [ ]:
from abc import ABC # abstract base class
class Tokenizer(ABC):
    """分词器的抽象基类"""
    # 定义了两个方法但无法直接调用，意思是子类必须自己实现。
    # 作用在于确定接口，保证之后写的分词器都有encode和decode方法
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError
    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError

from dataclasses import dataclass
@dataclass(frozen=True) 
# dataclass是来自模块dataclasses的装饰器。只写自动声明，可以自动生成__init__,__repr__,__eq__方法
# __init__ 进行类定义
# __repr__ 打印类属性
# __eq__ 相等判断
# @dataclass和需要装饰的类定义之间不能包含其他代码，但是可以空行
class BPETokenizerParams: # 定义一个“分词器参数”类
    """使用@dataclass语法糖快速定义BPETokenizerParams的两个实例属性：vocab和merges"""
    # 类实例化的时候需要传入两个参数：数据类型为dict[int, bytes]的字典，和数据类型为dict[tuple[int, int], int]的字典
    vocab:dict[int, bytes] # index -> bytes
    merges:dict[tuple[int, int], int] # index1, index2->new_index


from collections import defaultdict
def count_adjacent_pairs(indices:list[int])->dict[tuple[int, int], int]:
    """返回一个字典：把每个相邻token对和它们的出现次数相对应"""
    counts = defaultdict(int) # defaultdict访问key会在value不存在时赋值value=0
    for index1, index2 in zip(indices, indices[1:]):
        # zip(A,B)把两个列表按照位置配对，以长度短的列表为准
        # 如indices=[1,2,3,4]时，zip(indices, indices[1:])=[(1,2),(2,3),(3,4)]

        counts[(index1, index2)] += 1 
        # 注意元组可以作为字典的key，列表不行
        # 这一行代码的含义是：如果某个token对第一次出现，就从0变1，第二次就从1变2...
        # 去重是因为：counts是字典，它的key是唯一的，所以第二次遇到相等的token对会查到之前等值的key

    return counts

def merge(indices: list[int], pair: tuple[int, int], new_index: int)->list[int]:
    """merge函数把输入列表indices中所有相邻的token pair替换成new_index"""
    # indices是输入文本转换为的unicode编码
    # pair是程序找到的该轮出现次数最多的token对
    # new_index是程序记录的该token对所对应的新索引

    new_indices = []  
    i = 0  
    while i < len(indices):
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            # i + 1 < len(indices) : 当前位置不是最后一个token
            # 当前token是待替换token对的第一个token，下一个token是待替换token对的第二个token
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    return new_indices

def get_compression_ratio(string: str, indices: list[int]) -> float:  
    """Given `string` that has been tokenized into `indices`, return the number of UTF-8 bytes per token.."""
    num_bytes = len(bytes(string, encoding="utf-8"))  
    num_tokens = len(indices) # num_merges轮压缩后词表的长度                        
    return num_bytes / num_tokens


class BPETokenizer(Tokenizer):
    """BPE tokenizer given a set of merges and a vocabulary."""
    def __init__(self, params: BPETokenizerParams):
        # BPETokenizer继承Tokenizer抽象基类，实例化时接收训练好的参数
        self.params = params
    def encode(self, string: str) -> list[int]:
        indices = list(map(int, string.encode("utf-8")))  
        # Note: this is a very slow implementation
        for pair, new_index in self.params.merges.items():  
            # 按merges中记录的训练词表时的合并顺序依次合并unicode输入
            # .items()是一个字典方法，返回一对键值对
            indices = merge(indices, pair, new_index)  
        return indices
    def decode(self, indices: list[int]) -> str:
        """给一串分词后的token_id，按照训练好的词表把它翻译回对应的bytes，拼成一整段字节，最后还原为字符串"""
        bytes_list = list(map(self.params.vocab.get, indices)) 
        # map的第一个参数是传入了字典的get方法作为映射法则，get方法接收key返回value
        # 这里self.params.vocab的键值对是（int，bytes）
        # map返回值是一个迭代器，只能遍历一次，而且print出来是<map object ...>，看不到内容。用list转为普通列表，方便后续操作
        
        string = b"".join(bytes_list).decode("utf-8")  
        # b""是一个空bytes对象，作为分隔符
        # bytes类型的.joint方法和字符串.joint功能一样，可以用一个分隔符""把bytes_list的元素拼接为bytes串
        # .decode("utf-8")是bytes对象的decode方法，可以把bytes串按照指定编码（utf-8）翻译为字符串

        return string

### 细节知识点补充

In [2]:
print(list(map(int, "I like you".encode("utf-8"))))
print(list(map(int, "我喜欢你".encode("utf-8"))))
print(list(map(int, "\a".encode("utf-8"))))

bytes([255,0,1])

mydict = {('a','e'):4, ('c','d'):2}
print(max(mydict)) # 输出为('c','d')，因为只比较a和c的ASCII，c更大
print(max(mydict,key = lambda p: mydict[p])) # 输出为('a','e')，因为max函数按照筛选标准比较的是mydict的value值
print(max(mydict,key = mydict.get)) # 和lambda一样的比较方式，写法更简单

# 为什么需要@dataclass
# 
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __repr__(self):
        return f"Point({self.x},{self.y})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y
# 这个类定义中，只有x和y的类属性定义是有效信息，其余都是对类函数的重写，进行了大量无用功

# python中，双下划线包围的类方法叫double under method, dunder method，魔法方法。
# 不用于直接调用，而是给python解释器在特定时机自动调用


[73, 32, 108, 105, 107, 101, 32, 121, 111, 117]
[230, 136, 145, 229, 150, 156, 230, 172, 162, 228, 189, 160]
[7]
('c', 'd')
('a', 'e')
('a', 'e')


### 

In [ ]:
def train_bpe(string:str, num_merges:int) -> BPETokenizerParams:  
# 参数类型定义：输入文本string和要进行合并的次数num_merges，返回一个"分词器参数"对象

    indices = list(map(int, string.encode("utf-8")))
    # string.encode("utf-8")把字符串变成utf-8 编码
    # map(int, )对每个字节调用int()，即强制变为整数类型
    # list()把map的结果转化为列表
    # 例："the cat" -> [116, 104, 101, 32, 99, 97, 116]
    # 具体应用见前一个code cell

    merges:dict[tuple[int, int], int] = {}
    # merges = {} 
    # 空字典，记录每一次合并：把（左id，右id）这个pair合并为哪个id  
    # :dict[tuple[int, int], int] 为类型注释：merges这个变量，我期望它是一个字典，key是二元整数元组，value是int
    # 类型注释即使写错也不影响实际运行，只是帮助开发者理解变量和函数的预期类型，以及支持静态类型检查工具mypy进行类型验证

    vocab:dict[int, bytes] = {x: bytes([x]) for x in range(256)} # id -> bytes
    # 字典推导式，初始化256个token：
    # key是0到255，value是只含这一个字节的bytes对象

    for i in range(num_merges): # 循环num_merges次，每次进行一次合并
    
        counts = count_adjacent_pairs(indices) # 数每一token对出现的次数

        pair = max(counts, key = counts.get) 
        # 找到出现次数最多的token对
        # dict.get可以输入字典中存在的key，返回其对应的value。
        # max( , key )意为：按照这个给出的key，找出最大值对应的那个元素。注意key不是字典的键，而是max函数排序标准的关键字参数名。
        # 这里counts.get告诉max函数该用什么标准比较大小，找出最大的键值对：用对counts的每一个键进行counts.get的结果来比较大小。可以写成如下lambda形式：
        # pair = max(counts, key = lambda p: counts[p])

        new_index = 256 + i # 新token的id从256开始递增

        merges[pair] = new_index # 往之前定义的空字典merges里填东西：找到的pair合并成new_index

        vocab[new_index] = vocab[pair[0]]+vocab[pair[1]] 
        # 新token的字节表示是左右两半的字节表示的拼接
        # 比如pair[0]是116，之前由字典推导式定义的vocab中，vocab[116]是b't';同理vocab[pair[1]]是b'h'
        # new_index=255+1=256；vocab[new_index] = b't' + b'h' = b'th'，新的键值对被记录到字典vocab中
        
        indices = merge(indices, pair, new_index) # 把indices中所有相邻的(116,104)替换为256

    compression_ratio = get_compression_ratio(string, indices) # 计算压缩比

    return BPETokenizerParams(vocab = vocab, merges = merges),compression_ratio

mystr = "the cat in the hat"
params = train_bpe(mystr, num_merges = 4)
print({k: v for k, v in params[0].vocab.items() if k >= 256})
# params[0].vocab是字典。对字典调用.items()，会得到字典的(key,value)二元组
# 字典推导式每次循环，从params[0].vocab.items()里取出一对(key, value)，令k赋值为key，v赋值为value，当k大于255时收入新字典

print(train_bpe(mystr, num_merges = 3)[1])
print(train_bpe(mystr, num_merges = 4)[1])
print(train_bpe(mystr, num_merges = 5)[1]) # 压缩比随着压缩次数增加而升高

{256: b'th', 257: b'the', 258: b'the ', 259: b'at'}
1.5
1.8
2.0


{(116, 104): 256, (256, 101): 257, (257, 32): 258, (97, 116): 259}

In [ ]:
# 接下来可以使用这个分词器
tokenizer = BPETokenizer(params[0]) # 把train_bpe的输出结果，即训练好的分词器参数，作为分词器类的初始化参数
string = "the quick brown fox"
indices = tokenizer.encode(string) # 分词器类的编码方法，把unicode按照训练好的词表分词
reconstructed_string = tokenizer.decode(indices) # 分词器类的解码方法，indices复原为输入的string
assert string == reconstructed_string # assert是python关键字，用来做断言：“我坚信这里应该存在某个条件，若不满足就说明程序出了问题，需要报错停止”

indices # 可以看到，原始文本中的“the ”被词表替换为了258

[258, 113, 117, 105, 99, 107, 32, 98, 114, 111, 119, 110, 32, 102, 111, 120]